In [5]:
import sys
import os

sys.path.append(os.path.abspath("../")) 

import torch
from torch import nn
from torch.utils.data import DataLoader
from blocks import CatDogDataset, YoloLoss
from utils import get_detection_model, get_data_loaders, get_device, compute_mAP, \
                  decode_preds, decode_targets
import json
from tqdm import tqdm

In [6]:
device = get_device()

device

'mps'

In [7]:
def train_model(
        model, optimizer, criterion, train_loader, valid_loader, 
        max_epochs, num_classes, iou_threshold=0.5,
        scheduler=None, early_stopping=None, model_name="yolo_model", device="cuda"
    ):
    
    history = {
        "train_loss": [],
        "valid_loss": [],
        "valid_mAP": []  
    }
    
    n_epochs_without_impr = 0
    min_v_loss = float('inf')

    for epoch in range(max_epochs):
        model.train() 
        train_loss = 0
        
        train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{max_epochs} [Train]", leave=False)
        
        for X_batch, y_batch in train_loop:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            logits = model(X_batch)
            
            l, total_xy_loss, total_wh_loss, total_obj_loss, total_noobj_loss, total_class_loss = criterion(logits, y_batch)

            optimizer.zero_grad() 
            l.backward()   
               
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)    
            optimizer.step()    

            train_loss += l.item()
            train_loop.set_postfix(loss=l.item())

        model.eval() 
        valid_loss = 0
        
        all_epoch_preds = []
        all_epoch_targets = []
        
        valid_loop = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{max_epochs} [Valid]", leave=False)
        
        with torch.no_grad():
            for batch_idx, (X_valid, y_valid) in enumerate(valid_loop):
                X_valid, y_valid = X_valid.to(device), y_valid.to(device)

                v_logits = model(X_valid)
                v_l, _, _, _, _, _ = criterion(v_logits, y_valid)
                valid_loss += v_l.item()

                batch_preds = decode_preds(v_logits, conf_score_threshold=0.01, iou_threshold=0.4) 
                batch_targets = decode_targets(y_valid)
                
                batch_offset = batch_idx * X_valid.shape[0]
                
                for p in batch_preds:
                    p[0] += batch_offset 
                    all_epoch_preds.append(p)
                    
                for t in batch_targets:
                    t[0] += batch_offset 
                    all_epoch_targets.append(t)

        t_loss = train_loss / len(train_loader)
        v_loss = valid_loss / len(valid_loader)
        
        epoch_mAP = compute_mAP(all_epoch_preds, all_epoch_targets, iou_threshold, num_classes)

        print(f"Epoch [{epoch+1}/{max_epochs}] "
              f"| Train Loss: {t_loss:.4f} "
              f"| Valid Loss: {v_loss:.4f} "
              f"| Valid mAP: {epoch_mAP:.4f}")

        history["train_loss"].append(t_loss)
        history["valid_loss"].append(v_loss)
        history["valid_mAP"].append(epoch_mAP)

        current_lr = optimizer.param_groups[0]['lr']

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(v_loss)
            else:
                scheduler.step()

        new_lr = optimizer.param_groups[0]['lr']

        if new_lr < current_lr:
            print(f"--> [!] Learning rate decreased from {current_lr:.6f} to {new_lr:.6f}")

        if early_stopping is not None:
            if v_loss < min_v_loss:
                min_v_loss = v_loss
                n_epochs_without_impr = 0
                torch.save(model.state_dict(), f'../saved/weights/{model_name}.pth')
                torch.save(history, f'../saved/results/{model_name}_results.pth')
            else:
                n_epochs_without_impr += 1
                print(f"--> No improvement for {n_epochs_without_impr} epochs.")
            
            if n_epochs_without_impr >= early_stopping:
                print(f"Early stopping triggered at epoch {epoch+1}")
                return history
              
    return history

In [8]:
train_loader, valid_loader, test_loader = get_data_loaders()

S=7
B=2
C=37

model = get_detection_model(num_classes=C)
model = model.to(device)

lr = 0.0001 

optimizer = torch.optim.Adam(
    params=model.parameters(), 
    lr=lr, 
    weight_decay=1e-4
)

criterion = YoloLoss(S, B, C)

max_epochs = 100

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=4,
)

history = train_model(
    model, optimizer, criterion, train_loader, valid_loader, 
    max_epochs, C, iou_threshold=0.5,
    scheduler=scheduler, early_stopping=15, model_name="yolo_model", device=device
)

Epoch [1/100] | Train Loss: 1049.6490 | Valid Loss: 300.9313 | Valid mAP: 0.0000


Epoch [2/100] | Train Loss: 353.5910 | Valid Loss: 195.0964 | Valid mAP: 0.0000


Epoch [3/100] | Train Loss: 268.1479 | Valid Loss: 170.2849 | Valid mAP: 0.0000


Epoch [4/100] | Train Loss: 248.4682 | Valid Loss: 178.0175 | Valid mAP: 0.0000
--> No improvement for 1 epochs.


Epoch [5/100] | Train Loss: 232.4327 | Valid Loss: 152.3440 | Valid mAP: 0.0000


...

Epoch [96/100] | Train Loss: 22.8685 | Valid Loss: 54.9289 | Valid mAP: 0.6500
--> No improvement for 2 epochs.


Epoch [97/100] | Train Loss: 22.7413 | Valid Loss: 55.2967 | Valid mAP: 0.6405
--> No improvement for 3 epochs.


Epoch [98/100] | Train Loss: 22.9353 | Valid Loss: 55.0387 | Valid mAP: 0.6539
--> No improvement for 4 epochs.


Epoch [99/100] | Train Loss: 22.1697 | Valid Loss: 54.9422 | Valid mAP: 0.6636
--> [!] Learning rate decreased from 0.000025 to 0.000013
--> No improvement for 5 epochs.


Epoch [100/100] | Train Loss: 21.9182 | Valid Loss: 54.5940 | Valid mAP: 0.6648
--> No improvement for 6 epochs.
